In [213]:
# this file to get data from mongoDB
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv
import os
import json

In [214]:
# to get environment vars
load_dotenv()

True

In [215]:
# connect to the database
uri = os.getenv('MONGODB_CONNECTION_STRING')
client = MongoClient(uri, server_api=ServerApi('1'))

In [216]:
# Send a ping to confirm a successful connection
try:
  print("You successfully connected to MongoDB!")
except Exception as e:
  print(e)

You successfully connected to MongoDB!


In [217]:
db = client['Hotel'] # access database
collection = db['Hotel'] # access collection in a database

In [218]:
# create index on city field for faster querying
# collection.create_index({'city': 1})

# another one for title as it is queried frequently
# collection.create_index({'title': 1})

### Getting data for wordCloud

In [219]:
# get and concat all comments description
data1 = collection.aggregate([{
  '$project': {
    'comments.description': 1, 
    '_id': 0,
  },
}])

In [220]:
# print fetched data
text = ''
for obj in data1:
  for desc in obj['comments']:
    text += desc['description']
text

"We had a lovely stay at Serry Beach Resort. The property is beautiful, with a stunning pool and beach that made for the perfect relaxation spot. The atmosphere was peaceful, and the staff were friendly and welcoming. We truly enjoyed our time here...The service was top, and everyone was happy and friendly.\nFatima at the reception went out of her to see that we get the best room.\nThere was a variety of food dishes, so many options to choose. The best part was late breakfast and lunch for those...I loved how clean this property was, the great food, the accommodating staff, the entertainment timetable, the large single beds that could comfortably fit in two people, the clean beach, the view from my room… I could go on and onAmazing location at the beach with beautiful long pool and sun chairs. Breakfast offers large variety and is tasty.One of the best resorts in Hurghada recommend it to everyone going thereIt was clean, beautiful, scenic, peaceful, aesthetic, vibes were immaculate and

In [221]:
len(text)

527796

In [222]:
data2 = collection.aggregate([
  {
  '$match': {
    'city': {
        '$exists': True,
        '$ne': None
      }
    },
  },{
  '$group': {
    '_id': '$city', 
    'count': {
      '$sum': 1
    },
    'avg_rating': {
      '$avg': {
        '$ifNull': ['$rating', 0]
      }
    },
    'avg_price': {
      '$avg': {
        '$ifNull': ['$price', 0]
      }
    },
    'latitude': {
      '$avg': {
        '$ifNull': ['$latitude', 0]
      }
    },
    'longitude': {
      '$avg': {
        '$ifNull': ['$longitude', 0]
      }
    },
  },
}])

In [223]:
for obj in data2:
  print(obj)

{'_id': 'Ain Sokhna', 'count': 38, 'avg_rating': 6.321052631578947, 'avg_price': 737.9473684210526, 'latitude': 29.50768012344009, 'longitude': 32.453050041401916}
{'_id': 'Cairo', 'count': 52, 'avg_rating': 5.438461538461539, 'avg_price': 233.8846153846154, 'latitude': 30.047823623726966, 'longitude': 31.1401995173472}
{'_id': 'Fayoum', 'count': 28, 'avg_rating': 5.442857142857143, 'avg_price': 1062.0357142857142, 'latitude': 29.357198917220398, 'longitude': 30.644519366332357}
{'_id': 'Aswan', 'count': 50, 'avg_rating': 8.092, 'avg_price': 511.86, 'latitude': 24.089226775766093, 'longitude': 32.882503765847844}
{'_id': 'North Coast', 'count': 6, 'avg_rating': 9.166666666666666, 'avg_price': 3310.8333333333335, 'latitude': 30.962679019288046, 'longitude': 28.507248818801813}
{'_id': 'Dahab', 'count': 48, 'avg_rating': 8.25, 'avg_price': 343.5, 'latitude': 28.49722488437615, 'longitude': 34.51161868407057}
{'_id': 'Ismailia', 'count': 30, 'avg_rating': 4.6066666666666665, 'avg_price': 

In [224]:
data3 = collection.aggregate([
  {'$unwind': '$facilities'}, # deconstruct array elements
  {
    '$group': {
      '_id': '$facilities',
      'count': {
        '$sum': 1
      },
      'avg_rating': {
        '$avg': {
          '$ifNull': ['$rating', 0]
        }
      },
      'avg_price': {
        '$avg': {
          '$ifNull': ['$price', 0]
        } 
      }
    }
  }
])

In [225]:
for obj in data3:
  print(obj)

{'_id': 'Breakfast', 'count': 184, 'avg_rating': 6.8663043478260875, 'avg_price': 437.2717391304348}
{'_id': 'Basic free Wifi (12 Mbps)', 'count': 2, 'avg_rating': 8.0, 'avg_price': 0.0}
{'_id': 'Parking on site', 'count': 4, 'avg_rating': 0.0, 'avg_price': 0.0}
{'_id': 'Pool  – outdoor (kids)', 'count': 6, 'avg_rating': 8.633333333333333, 'avg_price': 0.0}
{'_id': '12 swimming pools', 'count': 4, 'avg_rating': 9.4, 'avg_price': 0.0}
{'_id': 'Excellent Breakfast', 'count': 34, 'avg_rating': 8.74705882352941, 'avg_price': 553.5294117647059}
{'_id': '5 restaurants', 'count': 4, 'avg_rating': 8.350000000000001, 'avg_price': 1025.25}
{'_id': '4 restaurants', 'count': 8, 'avg_rating': 6.25, 'avg_price': 0.0}
{'_id': '6 restaurants', 'count': 4, 'avg_rating': 9.3, 'avg_price': 0.0}
{'_id': 'Non-smoking rooms', 'count': 324, 'avg_rating': 7.6839506172839505, 'avg_price': 933.6697530864197}
{'_id': 'Airport shuttle (free)', 'count': 8, 'avg_rating': 7.75, 'avg_price': 1161.375}
{'_id': '2 rest

In [226]:
data4 = collection.aggregate([
  {'$project': {
      'rating_details': 1,
      'title': 1,
      '_id': 0
    }
  },
])

In [227]:
for obj in data4:
  print(obj)

{'title': 'شاليه في ستيلا مكادي الغردقة', 'rating_details': {}}
{'title': 'شاليه في ستيلا مكادي الغردقة', 'rating_details': {}}
{'title': 'Serry Beach Resort', 'rating_details': {'Staff': 8.8, 'Facilities': 9.0, 'Cleanliness': 9.2, 'Comfort': 9.2, 'Value for money': 8.3, 'Location': 9.3, 'Free Wifi': 9.4}}
{'title': 'Serry Beach Resort', 'rating_details': {'Staff': 8.8, 'Facilities': 9.0, 'Cleanliness': 9.2, 'Comfort': 9.2, 'Value for money': 8.3, 'Location': 9.3, 'Free Wifi': 9.4}}
{'title': 'Andalusia Blue Beach Hurghada', 'rating_details': {'Staff': 8.4, 'Facilities': 7.2, 'Cleanliness': 7.5, 'Comfort': 7.5, 'Value for money': 8.1, 'Location': 8.8, 'Free Wifi': 5.6}}
{'title': 'Andalusia Blue Beach Hurghada', 'rating_details': {'Staff': 8.4, 'Facilities': 7.2, 'Cleanliness': 7.5, 'Comfort': 7.5, 'Value for money': 8.1, 'Location': 8.8, 'Free Wifi': 5.6}}
{'title': 'براديس الاحياء', 'rating_details': {}}
{'title': 'براديس الاحياء', 'rating_details': {}}
{'title': 'B1-04 Juliana', 'ra

In [228]:
data5 = collection.aggregate([
  {
    '$group': {
      '_id': '$title',
      'avg_rating': {
        '$avg': {
          '$ifNull': ['$rating', 0]
        }
      },
      'avg_price': {
        '$avg': {
          '$ifNull': ['$price', 0]
        } 
      }
    }
  }
])

In [229]:
for obj in data5:
  print(obj)

{'_id': 'Blumar El Dome Hotel', 'avg_rating': 8.7, 'avg_price': 0.0}
{'_id': 'The Boutique Hotel Hurghada Marina', 'avg_rating': 8.2, 'avg_price': 0.0}
{'_id': 'Naama Bay Hotel & Resort', 'avg_rating': 8.4, 'avg_price': 0.0}
{'_id': 'Marina Residence Suites Port Ghalib', 'avg_rating': 8.6, 'avg_price': 0.0}
{'_id': 'شقق معاشات هيئه قناة السويس', 'avg_rating': 0.0, 'avg_price': 0.0}
{'_id': '4S Hotel Dahab', 'avg_rating': 8.1, 'avg_price': 0.0}
{'_id': 'whales camp dahab', 'avg_rating': 5.5, 'avg_price': 2040.5}
{'_id': 'Palma Hotel', 'avg_rating': 8.9, 'avg_price': 0.0}
{'_id': 'فيلا 6 غرف 6 حمام بحمام سباحه خاص هاسيندا باى الساحل الشمالى', 'avg_rating': 0.0, 'avg_price': 0.0}
{'_id': 'The G Einbay Golf Resort', 'avg_rating': 9.0, 'avg_price': 0.0}
{'_id': 'nuba nile hotel', 'avg_rating': 7.9, 'avg_price': 2130.5}
{'_id': 'bella vita- city center- criss resort', 'avg_rating': 5.0, 'avg_price': 0.0}
{'_id': 'Porto Said Resort & Spa', 'avg_rating': 8.3, 'avg_price': 0.0}
{'_id': 'Tolip E

In [230]:
data6 = collection.aggregate([
  {'$unwind': '$comments'},
  {
    '$group': {
      '_id': '$comments.country',
      'count': {
        '$sum': 1
      }
    }
  },
  {
    '$match': {
      '_id': {
        '$regex': r'\w{2,}'
      }
    }
  },
  {
    '$sort': {
      'count': 1
    }
  }
])

In [231]:
for obj in data6:
  print(obj)

{'_id': 'Costa Rica', 'count': 2}
{'_id': 'Trinidad and Tobago', 'count': 2}
{'_id': 'Sudan', 'count': 2}
{'_id': 'Malta', 'count': 2}
{'_id': 'Latvia', 'count': 2}
{'_id': 'Iraq', 'count': 2}
{'_id': 'Yemen', 'count': 2}
{'_id': 'Greece', 'count': 2}
{'_id': 'Lebanon', 'count': 2}
{'_id': 'Laos', 'count': 2}
{'_id': 'Georgia', 'count': 2}
{'_id': 'Libya', 'count': 2}
{'_id': 'Finland', 'count': 2}
{'_id': 'Morocco', 'count': 2}
{'_id': 'Ecuador', 'count': 2}
{'_id': 'Tunisia', 'count': 2}
{'_id': 'Estonia', 'count': 2}
{'_id': 'Azerbaijan', 'count': 2}
{'_id': 'Albania', 'count': 4}
{'_id': 'Norway', 'count': 4}
{'_id': 'Hong Kong', 'count': 4}
{'_id': 'Nigeria', 'count': 4}
{'_id': 'Ireland', 'count': 4}
{'_id': 'Serbia', 'count': 4}
{'_id': 'Algeria', 'count': 4}
{'_id': 'Brazil', 'count': 4}
{'_id': 'Panama', 'count': 4}
{'_id': 'Thailand', 'count': 4}
{'_id': 'Mexico', 'count': 4}
{'_id': 'Vietnam', 'count': 4}
{'_id': 'Moldova', 'count': 4}
{'_id': 'Philippines', 'count': 4}
{'_i

In [232]:
data7 = collection.aggregate([
  {
    '$match': {
      'price': {
        '$ne': None
      },
      'rating': {
        '$ne': None
      },
    }
  },
  {
    '$project': {
      '_id': 0,
      'rating': 1,
      'price': 1
    }
  }
])

In [233]:
for obj in data7:
  print(obj)

{'price': 1294.0, 'rating': 8.8}
{'price': 1744.0, 'rating': 8.8}
{'price': 1540.0, 'rating': 7.5}
{'price': 1437.0, 'rating': 7.5}
{'price': 739.0, 'rating': 10.0}
{'price': 1258.0, 'rating': 10.0}
{'price': 739.0, 'rating': 4.8}
{'price': 485.0, 'rating': 4.8}
{'price': 462.0, 'rating': 9.5}
{'price': 1109.0, 'rating': 9.5}
{'price': 2070.0, 'rating': 8.9}
{'price': 411.0, 'rating': 8.9}
{'price': 3593.0, 'rating': 8.0}
{'price': 1296.0, 'rating': 8.0}
{'price': 1294.0, 'rating': 8.8}
{'price': 1744.0, 'rating': 8.8}
{'price': 1540.0, 'rating': 7.5}
{'price': 1437.0, 'rating': 7.5}
{'price': 873.0, 'rating': 7.6}
{'price': 633.0, 'rating': 7.6}
{'price': 2160.0, 'rating': 8.8}
{'price': 5496.0, 'rating': 8.8}
{'price': 2703.0, 'rating': 8.4}
{'price': 1232.0, 'rating': 8.4}
{'price': 269.0, 'rating': 7.9}
{'price': 3832.0, 'rating': 7.9}
{'price': 7562.0, 'rating': 7.4}
{'price': 1540.0, 'rating': 7.4}
{'price': 4522.0, 'rating': 8.3}
{'price': 1183.0, 'rating': 8.3}
{'price': 6086.0

In [234]:
# to close this connection as Free clusters are limited
client.close()